# 01 - Exploratory Data Analysis: SPY Buy/Sell/Hold

**Goal of this notebook:** understand the data *before* modelling - feature distributions, correlations between
indicators, and the target-class distribution - and record every decision that will matter for leakage and imbalance.

**Contents:** 1. Data and chronological splits - 2. Stationarity: why returns, not prices - 3. Feature set -
4. Target definition and class balance - 5. Feature distributions - 6. Correlations - 7. Do features separate the
classes? - 8. Leakage audit - 9. Takeaways

In [ ]:
%matplotlib inline
import os, sys, warnings
from pathlib import Path
ROOT = Path.cwd().parent if Path.cwd().name == "notebooks" else Path.cwd()
sys.path.insert(0, str(ROOT / "src"))
warnings.filterwarnings("ignore")

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
pd.set_option("display.width", 140); pd.set_option("display.max_columns", 40)

from spy_signals import Config, plotting as P
from spy_signals.data import load_ohlcv

# SPY_FAST=1 shrinks the hyper-parameter grids; SPY_DATA_SOURCE=synthetic runs offline (smoke test only!)
cfg = Config(fast=os.environ.get("SPY_FAST") == "1")
ohlcv = load_ohlcv(cfg)

## 1. Data and chronological splits
SPY daily OHLCV (dividend/split-adjusted via `yfinance auto_adjust=True`). The shaded regions are fixed **in time order**; nothing is ever shuffled.

In [ ]:
display(ohlcv.head()); display(ohlcv.describe().T)
P.plot_price_with_splits(ohlcv, cfg);

## 2. Stationarity: why we convert prices to returns
A price series trends and its mean/variance change over time (non-stationary). A model fit on 2012 price *levels* has never
seen the levels of 2024. Returns and ratios fluctuate around a stable centre. The quick check below compares the mean of the
first and second half of the sample.

In [ ]:
half = len(ohlcv) // 2
r = ohlcv["Close"].pct_change().dropna()
pd.DataFrame({
    "Close (price level)": [ohlcv["Close"].iloc[:half].mean(), ohlcv["Close"].iloc[half:].mean()],
    "Daily return":        [r.iloc[:half].mean(), r.iloc[half:].mean()],
}, index=["first half mean", "second half mean"])

## 3. Feature set (all computed from data up to the close of day *t*)
| Group | Features |
|---|---|
| Returns | 1/5/10/21-day % return |
| Trend | close vs SMA10/SMA50, SMA50 vs SMA200, drawdown from 252-day high |
| Momentum | RSI(14), MACD line / signal / histogram (as % of price) |
| Bollinger | %B and band width (20, 2σ) |
| Volatility | rolling std of returns (10/21/63d), vol ratio, ATR(14)/price, high-low range |
| Context | close position in day range, overnight gap, volume vs 20-day average |

**Strict lagging:** the row for date *t* only uses OHLCV up to *t* and is used to predict what happens *after* *t*.

In [ ]:
from spy_signals.pipeline import prepare_data
from spy_signals.features import FEATURE_GROUPS
ds = prepare_data(cfg, ohlcv)
print({g: len(v) for g, v in FEATURE_GROUPS.items()}, "->", ds.X.shape[1], "features,", len(ds.X), "labelled rows")
ds.X.head()

## 4. Target definition and class balance
`fwd_ret[t] = Close[t+h] / Close[t] - 1`, then **Buy** if > +1 %, **Sell** if < -1 %, else **Hold** (h = 5 trading days).
The forward return looks into the future *by design* - it is the label, never a feature.

In [ ]:
from spy_signals.target import class_distribution
display(class_distribution(ds.y))
P.plot_forward_return_hist(ds.fwd_ret, cfg);

### Imbalance handling: how much does the threshold choice matter?
Widening the Hold band shifts mass to Hold; narrowing it creates more Buy/Sell. We keep ±1 % (assignment default) and rely on **class weights** in the models; this table shows the alternative lever.

In [ ]:
from spy_signals.target import make_target
rows = {}
for thr in (0.005, 0.0075, 0.01, 0.015, 0.02):
    lab = make_target(ohlcv["Close"], cfg.horizon, thr, thr)["label"].dropna()
    rows[f"±{thr:.2%}"] = {"Sell": (lab == -1).mean(), "Hold": (lab == 0).mean(), "Buy": (lab == 1).mean()}
thr_table = pd.DataFrame(rows).T
display(thr_table.round(3))
P.plot_threshold_sensitivity(thr_table);

In [ ]:
from spy_signals.splits import chronological_split
sp = chronological_split(ds.X, ds.y, cfg)
display(sp.summary())
print(f"Purged rows (labels would peek into the next split): train={sp.purged_train}, validation={sp.purged_val}")
P.plot_class_distribution({"train": sp.y_train, "validation": sp.y_val, "test": sp.y_test});

**Reading this:** Hold is usually the largest class and Sell the smallest, because the market drifts upward. A model that always says the majority class already scores well on accuracy - so we will report **balanced accuracy and macro-F1** and compare against a majority baseline.

## 5. Feature distributions
Heavy tails (volatility, volume, returns) are normal for financial data. Logistic regression gets a `StandardScaler`; trees do not need scaling.

In [ ]:
P.plot_feature_distributions(ds.X);

## 6. Correlations between indicators
Many indicators are different views of the same thing (e.g. three volatility windows). Correlated features share credit in importance tables, which we account for when interpreting.

In [ ]:
P.plot_correlation(ds.X);
c = ds.X.corr().abs()
pairs = (c.where(np.triu(np.ones(c.shape), 1).astype(bool)).stack().sort_values(ascending=False))
print("Most correlated feature pairs (|r|):"); pairs.head(10).round(3)

## 7. Do the features separate the classes at all?
Mean z-score of each feature within each class. If the three bars per feature are nearly equal, the feature carries little information about the label.

In [ ]:
P.plot_feature_vs_target(ds.X, ds.y);
fwd_corr = ds.X.corrwith(ds.fwd_ret).sort_values(key=abs, ascending=False)
print("Largest |correlation| between a feature and the FUTURE return (expect small):"); fwd_corr.head(8).round(3)

## 8. Leakage audit (run before any modelling)

In [ ]:
from spy_signals import audit
checks = [audit.audit_no_lookahead(ohlcv), audit.audit_target_not_in_features(ds.X, ds.fwd_ret),
          audit.audit_split_order(ds.X, sp, ohlcv.index, cfg.horizon)]
audit.format_audits(checks)

## 9. Takeaways to carry into modelling
1. Use returns/ratios only (stationary); no raw price levels.
2. Classes are imbalanced -> `class_weight='balanced'` and judge with balanced accuracy / macro-F1, not plain accuracy.
3. Features are weakly related to the label and to each other in clusters -> expect modest scores (~35-40 % accuracy is normal) and be suspicious of anything much higher.
4. Labels overlap across consecutive days -> purge `horizon` rows at each split boundary and use `TimeSeriesSplit(gap=horizon)`.